# 01 — Exploratory Data Analysis (EDA)

**Dataset:** DPWH Flood Control Projects (2018–2025)  
**Rows:** ~9,856  
**Goal:** Understand the dataset structure, identify data quality issues, and visualize key patterns.

In [ ]:
# === Install dependencies (for Google Colab) ===
# !pip install pandas matplotlib seaborn plotly

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

## 1. Load the Dataset

In [ ]:
# For Google Colab: upload the CSV first or mount Google Drive
# from google.colab import files
# uploaded = files.upload()

df = pd.read_csv('../data/dpwh_flood_control_projects.csv')
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 2. Data Types and Info

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

## 3. Missing Values Analysis

In [ ]:
null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df) * 100).round(2)

null_report = pd.DataFrame({
    'Null Count': null_counts,
    'Null %': null_pct
}).sort_values('Null Count', ascending=False)

print('=== Missing Values Report ===')
null_report[null_report['Null Count'] > 0]

In [ ]:
# Visualize missing values
fig, ax = plt.subplots(figsize=(14, 5))
cols_with_nulls = null_report[null_report['Null Count'] > 0].index
if len(cols_with_nulls) > 0:
    sns.barplot(x=cols_with_nulls, y=null_report.loc[cols_with_nulls, 'Null %'], ax=ax)
    ax.set_title('Missing Values by Column (%)', fontsize=16, fontweight='bold')
    ax.set_ylabel('Percentage Missing')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('No missing values found!')

## 4. Parse Dates and Numeric Columns

In [ ]:
# Convert date columns
df['ActualCompletionDate'] = pd.to_datetime(df['ActualCompletionDate'], errors='coerce')
df['StartDate'] = pd.to_datetime(df['StartDate'], errors='coerce')

# Ensure numeric columns
df['ApprovedBudgetForContract'] = pd.to_numeric(df['ApprovedBudgetForContract'], errors='coerce')
df['ContractCost'] = pd.to_numeric(df['ContractCost'], errors='coerce')

# Compute derived features
df['DurationDays'] = (df['ActualCompletionDate'] - df['StartDate']).dt.days
df['BudgetDifference'] = df['ContractCost'] - df['ApprovedBudgetForContract']
df['BudgetRatio'] = df['ContractCost'] / df['ApprovedBudgetForContract']

print(f'Date range: {df["StartDate"].min()} to {df["ActualCompletionDate"].max()}')
print(f'Duration stats (days):')
df['DurationDays'].describe()

## 5. Province Distribution

In [ ]:
province_counts = df['Province'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(14, 7))
sns.barplot(x=province_counts.values, y=province_counts.index, ax=ax, palette='viridis')
ax.set_title('Top 20 Provinces by Project Count', fontsize=16, fontweight='bold')
ax.set_xlabel('Number of Projects')
ax.set_ylabel('Province')
plt.tight_layout()
plt.show()

print(f'Total unique provinces: {df["Province"].nunique()}')

## 6. Yearly Spending Trends

In [ ]:
yearly_stats = df.groupby('FundingYear').agg(
    total_projects=('ProjectId', 'count'),
    total_budget=('ApprovedBudgetForContract', 'sum'),
    total_cost=('ContractCost', 'sum'),
    avg_cost=('ContractCost', 'mean'),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Project count by year
sns.barplot(data=yearly_stats, x='FundingYear', y='total_projects', ax=axes[0], palette='viridis')
axes[0].set_title('Projects per Year', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Count')

# Total spending by year
sns.barplot(data=yearly_stats, x='FundingYear', y='total_cost', ax=axes[1], palette='magma')
axes[1].set_title('Total Contract Cost per Year (PHP)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Total Cost (PHP)')
axes[1].ticklabel_format(style='plain', axis='y')

plt.tight_layout()
plt.show()

yearly_stats

## 7. Contractor Frequency

In [ ]:
contractor_counts = df['Contractor'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(14, 7))
sns.barplot(x=contractor_counts.values, y=contractor_counts.index, ax=ax, palette='rocket')
ax.set_title('Top 15 Contractors by Project Count', fontsize=16, fontweight='bold')
ax.set_xlabel('Number of Projects')
ax.set_ylabel('Contractor')
plt.tight_layout()
plt.show()

print(f'Total unique contractors: {df["Contractor"].nunique()}')

## 8. Project Duration Distribution

In [ ]:
valid_duration = df['DurationDays'].dropna()
valid_duration = valid_duration[(valid_duration > 0) & (valid_duration < 2000)]

fig, ax = plt.subplots(figsize=(14, 6))
sns.histplot(valid_duration, bins=50, kde=True, ax=ax, color='#2ecc71')
ax.set_title('Project Duration Distribution (Days)', fontsize=16, fontweight='bold')
ax.set_xlabel('Duration (Days)')
ax.set_ylabel('Frequency')
ax.axvline(valid_duration.median(), color='red', linestyle='--', label=f'Median: {valid_duration.median():.0f} days')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Budget Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Approved Budget
budget = df['ApprovedBudgetForContract'].dropna()
sns.histplot(budget / 1e6, bins=50, kde=True, ax=axes[0], color='#3498db')
axes[0].set_title('Approved Budget Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Budget (Million PHP)')

# Contract Cost
cost = df['ContractCost'].dropna()
sns.histplot(cost / 1e6, bins=50, kde=True, ax=axes[1], color='#e74c3c')
axes[1].set_title('Contract Cost Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Cost (Million PHP)')

plt.tight_layout()
plt.show()

## 10. Type of Work Breakdown

In [ ]:
type_counts = df['TypeOfWork'].value_counts()

fig, ax = plt.subplots(figsize=(14, 5))
sns.barplot(x=type_counts.values, y=type_counts.index, ax=ax, palette='coolwarm')
ax.set_title('Projects by Type of Work', fontsize=16, fontweight='bold')
ax.set_xlabel('Count')
plt.tight_layout()
plt.show()

## 11. Region Distribution

In [ ]:
region_counts = df['Region'].value_counts()

fig, ax = plt.subplots(figsize=(14, 7))
sns.barplot(x=region_counts.values, y=region_counts.index, ax=ax, palette='crest')
ax.set_title('Projects by Region', fontsize=16, fontweight='bold')
ax.set_xlabel('Count')
plt.tight_layout()
plt.show()

## 12. Geographic Scatter Plot

In [ ]:
geo = df.dropna(subset=['ProjectLatitude', 'ProjectLongitude'])

fig = px.scatter_mapbox(
    geo,
    lat='ProjectLatitude',
    lon='ProjectLongitude',
    color='Region',
    hover_name='ProjectName',
    hover_data=['Province', 'ContractCost', 'FundingYear'],
    title='DPWH Flood Control Projects — Nationwide',
    mapbox_style='carto-positron',
    zoom=5,
    center={'lat': 12.8, 'lon': 121.7},
    height=700,
    opacity=0.6,
)
fig.show()

## 13. Correlation Heatmap (Numeric Features)

In [ ]:
numeric_cols = ['ApprovedBudgetForContract', 'ContractCost', 'ContractorCount',
                'FundingYear', 'DurationDays', 'BudgetDifference', 'BudgetRatio']

corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 14. Key Findings Summary

Document your key observations here after running the notebook:

- **Total projects:** ~9,856
- **Date range:** 2018–2025
- **Missing values:** (fill in after running)
- **Top provinces:** (fill in)
- **Most common project type:** (fill in)
- **Average project duration:** (fill in)
- **Average contract cost:** (fill in)